# K-Nearest Neighbors Implementation

## Overview

In this notebook, I applied **K-Nearest Neighbors** to predict Big Five personality traits (Extraversion, Agreeableness, Conscientiousness, Neuroticism, Openness) from textual life narrative responses (Q1–Q32).

## What is K-Nearest Neighbors?

**KNN** is a non-parametric algorithm that predicts a value by looking at the **K most similar training examples** and averaging their target values (for regression).

For a new narrative, KNN:
1. Computes the distance between that narrative's TF-IDF vector and all training narratives
2. Finds the K closest neighbors
3. Predicts the trait score as the **weighted average** of those K neighbors' scores

### Why KNN for personality prediction?
- Intuitive: people with similar narratives likely have similar personalities
- No assumptions about the functional form of the relationship
- Works as a useful baseline to compare against more complex models
- `weights='distance'` gives closer neighbors more influence

### Caveat
KNN suffers from the **curse of dimensionality** — in high-dimensional TF-IDF space, all points become equidistant. We use TruncatedSVD to reduce dimensions first.

### Pipeline
```
Narrative text (Q columns)
        ↓
TF-IDF Vectorization
        ↓
TruncatedSVD → 100 dense dimensions
        ↓
KNeighborsRegressor (one per trait via MultiOutputRegressor)
        ↓
5-Fold Cross-Validation + Evaluation
```

## 1. Data preprocessing, import required packages

As specified in the random forest module, all narrative columns are combined into a single text string per participant, then apply **TF-IDF** (Term Frequency–Inverse Document Frequency) to convert text into a numeric matrix. 

The key process of supervised learning is for model to learn patterns from training set and evaluate model performance on unseen test set. Thus, data is split into **80% training** and **20% testing**.`random_state=42` ensures reproducibility

In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

# Find repo root automatically
repo_root = Path.cwd().resolve()
while not (repo_root / "BFI_2_life_narative_metadata.json").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find BFI_2_life_narative_metadata.json")
    repo_root = repo_root.parent

# Load data
with open(repo_root / "BFI_2_life_narative_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(repo_root / "BFI_2_life_narrative.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

df = pd.DataFrame(responses)

# Reverse-code BFI items
reverse_items = metadata["reverse_code_items"]

for item in reverse_items:
    if item in df.columns:
        df[item] = 6 - df[item]

# Big Five trait means
traits = {
    key: value
    for key, value in metadata.items()
    if (
        isinstance(value, list)
        and value
        and isinstance(value[0], str)
        and value[0].startswith("Item")
        and not key.startswith(("Item", "Q", "CWB", "OCB"))
        and key != "reverse_code_items"
    )
}

for trait, items in traits.items():
    available_items = [item for item in items if item in df.columns]
    df[trait] = df[available_items].mean(axis=1)

# CWB and OCB means
cwb_cols = [f"CWB{i}" for i in range(1, 11) if f"CWB{i}" in df.columns]
ocb_cols = [f"OCB{i}" for i in range(1, 11) if f"OCB{i}" in df.columns]

df["CWB"] = df[cwb_cols].mean(axis=1)
df["OCB"] = df[ocb_cols].mean(axis=1)

# Text features
q_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("Q")]

X_text = df[q_cols].copy()
X_all = X_text.fillna("").agg(" ".join, axis=1)

# Targets
big_five = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

y = df[big_five].copy()

print("✓ Data preprocessing complete")
print("Repo root:", repo_root)
print("df shape:", df.shape)
print("X_text shape:", X_text.shape)
print("X_all shape:", X_all.shape)
print(y.describe().round(3))

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

✓ Imports complete
✓ Data preprocessing complete
Repo root: /Users/cindy/cmor438_Spring2026/cmor438_Spring2026
df shape: (500, 138)
X_text shape: (500, 32)
X_all shape: (500,)
       Extraversion  Agreeableness  Conscientiousness  Neuroticism  Openness
count       500.000        500.000            500.000      500.000   500.000
mean          3.177          3.786              3.642        2.879     3.833
std           0.761          0.576              0.710        0.846     0.618
min           1.000          2.000              1.083        1.000     1.750
25%           2.667          3.417              3.167        2.333     3.417
50%           3.167          3.833              3.750        2.917     3.833
75%           3.750          4.167              4.083        3.417     4.250
max           5.000          5.000              5.000        4.917     5.000
Training samples : 400
Test samples     : 100


In [3]:
def cross_validate_model(model, X, y, trait_cols, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_results = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        for j, trait in enumerate(trait_cols):
            y_true = y_te[trait].to_numpy()
            y_pred = pred[:, j]
            fold_results.append({
                "Fold": fold + 1, "Trait": trait,
                "MAE": mean_absolute_error(y_true, y_pred),
                "R2":  r2_score(y_true, y_pred),
                "Pearson_r": np.corrcoef(y_true, y_pred)[0, 1]
            })
    results_df = pd.DataFrame(fold_results)
    summary    = results_df.groupby("Trait").mean(numeric_only=True).round(4)
    return results_df, summary


def plot_cv_summary(summary, title):
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(summary.index, summary["R2"], color=colors)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylabel("Mean CV R²")
    ax.set_title(title)
    for bar, val in zip(bars, summary["R2"]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()

print("✓ Helper functions defined")

✓ Helper functions defined


## 2. Tune K with GridSearchCV

The most important hyperparameter in KNN is **K** (number of neighbors):
- **Small K** (e.g., K=1): very sensitive to noise, overfits
- **Large K** (e.g., K=20): smoother predictions, may underfit

We search over K = 3, 5, 7, 10, 15 using Extraversion as a representative trait.

In [6]:
param_grid = {"knn__n_neighbors": [10, 15, 20, 25, 30, 35, 40, 50]}

best_k = {}

for trait in big_five:
    gs_pipe = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=5, max_features=100_000)),
        ("svd",   TruncatedSVD(n_components=100, random_state=42)),
        ("knn",   KNeighborsRegressor(weights="distance"))
    ])
    gs = GridSearchCV(gs_pipe, param_grid, cv=5, scoring="r2", n_jobs=-1)
    gs.fit(X_train, y_train[trait])
    best_k[trait] = gs.best_params_["knn__n_neighbors"]
    print(f"  {trait:20s} best K={best_k[trait]}  CV R²={gs.best_score_:.4f}")

print("\nBest K per trait:", best_k)

/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:54

  Extraversion         best K=35  CV R²=0.0266


/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:54

  Agreeableness        best K=50  CV R²=-0.0060


/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:54

  Conscientiousness    best K=20  CV R²=0.0366


/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:54

  Neuroticism          best K=15  CV R²=-0.0231


/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:54

  Openness             best K=50  CV R²=-0.0031

Best K per trait: {'Extraversion': 35, 'Agreeableness': 50, 'Conscientiousness': 20, 'Neuroticism': 15, 'Openness': 50}


In [8]:
final_knn_pipelines = {}

for trait in big_five:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=5, max_features=100_000)),
        ("svd",   TruncatedSVD(n_components=100, random_state=42)),
        ("knn",   KNeighborsRegressor(n_neighbors=best_k[trait], weights="distance"))
    ])
    pipe.fit(X_train, y_train[trait])
    final_knn_pipelines[trait] = pipe
    print(f"✓ Trained {trait} with K={best_k[trait]}")

# Test set evaluation
rows = []
for trait in big_five:
    y_pred_trait = final_knn_pipelines[trait].predict(X_test)
    r2  = r2_score(y_test[trait], y_pred_trait)
    mae = mean_absolute_error(y_test[trait], y_pred_trait)
    rows.append({"Trait": trait, "R²": round(r2, 4), "MAE": round(mae, 4)})

results_knn = pd.DataFrame(rows).set_index("Trait")
print("\nKNN — Test Set Performance")
print(results_knn)

✓ Trained Extraversion with K=35
✓ Trained Agreeableness with K=50
✓ Trained Conscientiousness with K=20
✓ Trained Neuroticism with K=15
✓ Trained Openness with K=50

KNN — Test Set Performance
                       R²     MAE
Trait                            
Extraversion       0.0157  0.6348
Agreeableness     -0.0452  0.4067
Conscientiousness -0.0275  0.6234
Neuroticism       -0.0317  0.7432
Openness           0.0341  0.5340


In [9]:
X_all_knn = df[q_cols].fillna("").agg(" ".join, axis=1).reset_index(drop=True)
y_cv      = y.reset_index(drop=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
print("KNN — 5-Fold Cross-Validation")
print("-" * 45)

cv_rows = []
for trait in big_five:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=5, max_features=100_000)),
        ("svd",   TruncatedSVD(n_components=100, random_state=42)),
        ("knn",   KNeighborsRegressor(n_neighbors=best_k[trait], weights="distance"))
    ])

    fold_r2, fold_mae, fold_r = [], [], []
    for train_idx, test_idx in kf.split(X_all_knn):
        X_tr = X_all_knn.iloc[train_idx]; X_te = X_all_knn.iloc[test_idx]
        y_tr = y_cv[trait].iloc[train_idx]; y_te = y_cv[trait].iloc[test_idx]
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)
        fold_r2.append(r2_score(y_te, y_pred))
        fold_mae.append(mean_absolute_error(y_te, y_pred))
        fold_r.append(np.corrcoef(y_te, y_pred)[0, 1])

    cv_rows.append({
        "Trait":     trait,
        "CV Mean R²": round(np.mean(fold_r2),  4),
        "CV Std R²":  round(np.std(fold_r2),   4),
        "CV MAE":     round(np.mean(fold_mae),  4),
        "Pearson r":  round(np.mean(fold_r),    4),
    })
    print(f"  {trait:20s} R²={np.mean(fold_r2):.4f} ± {np.std(fold_r2):.4f}  "
          f"MAE={np.mean(fold_mae):.4f}  r={np.mean(fold_r):.4f}")

cv_results_knn = pd.DataFrame(cv_rows).set_index("Trait")
print("\nSummary:")
print(cv_results_knn)

KNN — 5-Fold Cross-Validation
---------------------------------------------
  Extraversion         R²=0.0207 ± 0.0449  MAE=0.6050  r=0.2233
  Agreeableness        R²=-0.0255 ± 0.0138  MAE=0.4600  r=0.0046
  Conscientiousness    R²=0.0151 ± 0.0364  MAE=0.5707  r=0.1855
  Neuroticism          R²=-0.0318 ± 0.0837  MAE=0.6801  r=0.2039
  Openness             R²=0.0109 ± 0.0146  MAE=0.4897  r=0.1672

Summary:
                   CV Mean R²  CV Std R²  CV MAE  Pearson r
Trait                                                      
Extraversion           0.0207     0.0449  0.6050     0.2233
Agreeableness         -0.0255     0.0138  0.4600     0.0046
Conscientiousness      0.0151     0.0364  0.5707     0.1855
Neuroticism           -0.0318     0.0837  0.6801     0.2039
Openness               0.0109     0.0146  0.4897     0.1672


## Results analysis

#### Pearson's r Comparison

| Trait | KNN | Decision Tree | Random Forest | Gradient Boosting |
|---|---:|---:|---:|---:|
| Extraversion | 0.2233 | -0.0036 | **0.2981** | 0.1857 |
| Agreeableness | 0.0046 | -0.0138 | **0.0296** | 0.0197 |
| Conscientiousness | 0.1855 | 0.2035 | **0.2916** | 0.2730 |
| Neuroticism | 0.2039 | 0.1248 | **0.2601** | 0.2551 |
| Openness | **0.1672** | 0.0436 | 0.1396 | 0.1591 |

KNN Outperformed the Single Decision Tree all traits except conscientiousness, suggesting that similarity-based prediction was more effective than relying on a single tree split structure. The best tree-based method (random forest) has stronger overall prediction than KNN, indicating that aggregating many trees captured personality-related text patterns better than local distance matching. The prediction performance was on par for Gradient Boosting and KNN, with no single approach dominated across all traits.

KNN had the highest Pearson's r for **Openness** compared with the tree-based models. This may mean that people with similar language styles or topic interests tend to resemble each other in openness, making neighbor-based prediction especially useful for that trait.

This pattern shows that in text data:

- KNN can capture local similarity between narratives
- But sparse high-dimensional text features often make distance measures noisy
- Ensemble methods are usually more robust to this complexity